In [1]:
!pip install matplotlib numpy ipywidgets

In [2]:
# Kód pro Midpoint algoritmus kružnice

def midpoint_circle_didactic(r):
    """
    Midpoint algoritmus pro rasterizaci kružnice se středem v [0, 0] a poloměrem r.
    Počítá 1. oktant (od [0, r] směrem k x = y) a využívá osmičetnou symetrii.
    """
    x = 0
    y = r
    
    # Počáteční hodnota rozhodovací proměnné (prediktoru) P0 = 1 - r
    P = 1 - r

    steps = []
    all_drawn_pixels = []

    krok = 0
    while x <= y:
        # 8 symetrických bodů pro aktuální (x, y)
        symetricke_body = [
            ( x,  y), ( y,  x),
            ( y, -x), ( x, -y),
            (-x, -y), (-y, -x),
            (-y,  x), (-x,  y)
        ]
        
        for pt in symetricke_body:
            if pt not in all_drawn_pixels:
                all_drawn_pixels.append(pt)

        P_current = P

        # Rozhodovací logika pro další krok v hlavním oktantu
        if P < 0:
            # Středový bod je uvnitř kružnice -> Y se nemění
            P_next = P + 2 * x + 3
            next_y = y
        else:
            # Středový bod je vně kružnice -> Y klesne o 1
            P_next = P + 2 * (x - y) + 5
            next_y = y - 1

        steps.append({
            "krok": krok,
            "hlavni_pixel": (x, y),
            "aktualni_symetrie": symetricke_body,
            "vykreslene_pixely": list(all_drawn_pixels),
            "P_pred": P_current,
            "P_po": P_next,
            "dalsi_hlavni": (x + 1, next_y)
        })

        x += 1
        y = next_y
        P = P_next
        krok += 1

    return steps

In [3]:
# Kód pro vykreslení algoritmu do grafu

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display

# Pevně daný poloměr kružnice (střed v [0, 0])
POLOMER = 8
historie = midpoint_circle_didactic(POLOMER)
max_krok = len(historie) - 1

aktualni_krok = 0

# Vytvoření ovládacích prvků
btn_zpet = widgets.Button(description='<- Krok zpět', disabled=True, button_style='info')
btn_vpred = widgets.Button(description='Krok vpřed ->', disabled=False, button_style='info')
lbl_stav = widgets.Label(value=f"Krok: 0 / {max_krok}")
vystup_grafu = widgets.Output(layout=widgets.Layout(min_height='680px'))

def vykresli_aktualni_stav():
    data_kroku = historie[aktualni_krok]
    cur_x, cur_y = data_kroku["hlavni_pixel"]
    P_pred = data_kroku["P_pred"]
    P_po = data_kroku["P_po"]
    sym_body = data_kroku["aktualni_symetrie"]

    limit = POLOMER + 2

    # Popis nad graf
    if aktualni_krok == max_krok:
        nadpis = (
            f"Krok {aktualni_krok} z {max_krok} | Hlavní pixel [{cur_x}, {cur_y}] (dosáhli jsme hranice oktantu X ≥ Y)\n"
            f"Díky osmičetné symetrii je celá kružnice kompletně vykreslená!"
        )
    else:
        if P_pred < 0:
            vysvetleni = (
                f"Prediktor P je záporný ({P_pred} < 0) → Středový bod leží UVNITŘ ideální kružnice.\n"
                f"Rozhodnutí: Jdeme ROVNĚ DOPRAVA na pixel [{cur_x + 1}, {cur_y}] (výška Y zůstává {cur_y}). "
                f"Nový prediktor P = {P_po}."
            )
        else:
            vysvetleni = (
                f"Prediktor P je kladný nebo nula ({P_pred} ≥ 0) → Středový bod leží VNĚ ideální kružnice.\n"
                f"Rozhodnutí: Jdeme ŠIKMO DOLŮ na pixel [{cur_x + 1}, {cur_y - 1}] (výška Y klesne na {cur_y - 1}). "
                f"Nový prediktor P = {P_po}."
            )
        nadpis = f"Krok {aktualni_krok} z {max_krok} | Počítáme hlavní pixel [{cur_x}, {cur_y}] + 7 symetrických bodů\n" + vysvetleni

    with vystup_grafu:
        vystup_grafu.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.5, 8.5))

        # 1. Vykreslení již rasterizovaných pixelů
        for px, py in data_kroku["vykreslene_pixely"]:
            if (px, py) == (cur_x, cur_y):
                barva = '#ff6b6b'  # Hlavní počítaný pixel v 1. oktantu (červená)
                alfa = 0.95
            elif (px, py) in sym_body:
                barva = '#ffd166'  # Symetrické pixely přidané v tomto kroku (žlutá)
                alfa = 0.85
            else:
                barva = '#4ecdc4'  # Starší hotové pixely (tyrkysová)
                alfa = 0.45

            rect = patches.Rectangle((px - 0.5, py - 0.5), 1, 1, 
                                     linewidth=1.2, edgecolor='black', 
                                     facecolor=barva, alpha=alfa, zorder=2)
            ax.add_patch(rect)

        # 2. Vykreslení os symetrie (rozdělení na 8 oktantů)
        ax.axhline(0, color='#adb5bd', linestyle='-', linewidth=1, zorder=1)
        ax.axvline(0, color='#adb5bd', linestyle='-', linewidth=1, zorder=1)
        ax.plot([-limit, limit], [-limit, limit], color='#adb5bd', linestyle=':', linewidth=1, zorder=1, label='Osy symetrie (8 oktantů)')
        ax.plot([-limit, limit], [limit, -limit], color='#adb5bd', linestyle=':', linewidth=1, zorder=1)

        # 3. Ideální matematická kružnice
        uhly = np.linspace(0, 2 * np.pi, 200)
        ax.plot(POLOMER * np.cos(uhly), POLOMER * np.sin(uhly), 
                color='#1a535c', linestyle='--', linewidth=2, label='Ideální kružnice', zorder=3)

        # 4. Zvýraznění hlavního pixelu, symetrických bodů a kandidátů
        ax.scatter([cur_x], [cur_y], color='red', s=45, zorder=5, label=f'Hlavní počítaný pixel [{cur_x}, {cur_y}]')
        
        # Ukázka v легенdě pro žluté symetrické pixely
        ax.scatter([], [], color='#ffd166', edgecolor='black', marker='s', s=80, label='Symetricky zrcadlené pixely (7×)')

        if aktualni_krok < max_krok:
            ax.scatter([cur_x + 1, cur_x + 1], [cur_y, cur_y - 1], 
                       color='orange', edgecolor='black', s=80, zorder=5, label='Kandidáti na další vykreslení')

        # Nastavení mřížky
        ax.set_xticks(range(-limit, limit + 1, 2))
        ax.set_yticks(range(-limit, limit + 1, 2))
        ax.set_xticks([x - 0.5 for x in range(-limit, limit + 1)], minor=True)
        ax.set_yticks([y - 0.5 for y in range(-limit, limit + 1)], minor=True)

        ax.grid(which='minor', color='#cccccc', linestyle='-', linewidth=0.8)
        ax.set_xlim(-limit - 0.5, limit + 0.5)
        ax.set_ylim(-limit - 0.5, limit + 0.5)
        ax.set_aspect('equal')

        # Panel nad grafem
        ax.set_title(
            nadpis, 
            fontsize=10.5, pad=14, loc='left',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8f9fa', edgecolor='#dee2e6')
        )

        ax.legend(loc='upper left', fontsize=9)
        plt.tight_layout()
        plt.show()

# Obsluha kliknutí na tlačítka
def pri_kliknuti_zpet(b):
    global aktualni_krok
    if aktualni_krok > 0:
        aktualni_krok -= 1
        aktualizuj_ui()

def pri_kliknuti_vpred(b):
    global aktualni_krok
    if aktualni_krok < max_krok:
        aktualni_krok += 1
        aktualizuj_ui()

def aktualizuj_ui():
    btn_zpet.disabled = (aktualni_krok == 0)
    btn_vpred.disabled = (aktualni_krok == max_krok)
    lbl_stav.value = f"Krok: {aktualni_krok} / {max_krok}"
    vykresli_aktualni_stav()

btn_zpet.on_click(pri_kliknuti_zpet)
btn_vpred.on_click(pri_kliknuti_vpred)

# Složení panelu a vykreslení
ovladaci_panel = widgets.HBox([btn_zpet, btn_vpred, lbl_stav])
#display(vystup_grafu, ovladaci_panel)
#vykresli_aktualni_stav()

## Princip a pseudokód Midpoint algoritmu pro kružnici

**Midpoint (středový) algoritmus** vykresluje kružnici pomocí celočíselné aritmetiky a využívá její **osmičetnou symetrii**. Algoritmus počítá souřadnice pouze v jednom oktantu (od vrcholu $[0, R]$ směrem doprava dolů, dokud platí $X \le Y$). V každém kroku se souřadnice $X$ zvýší o 1 a pomocí prediktoru $P$ se testuje, zda střed mezi dvěma kandidátními pixely leží uvnitř ($P < 0$, výška $Y$ se nemění), nebo vně kružnice ($P \ge 0$, výška $Y$ klesne o 1). Zbylých 7 symetrických bodů se vykreslí automaticky zrcadlením.

```text
ALGORITMUS: Midpoint algoritmus kružnice
VSTUP:  Poloměr kružnice R (střed v počátku [0, 0])
VÝSTUP: Vykreslené pixely kružnice

1. Nastav výchozí bod na vrcholu kružnice a počáteční prediktor:
   x = 0
   y = R
   P = 1 - R

2. Dokud platí x <= y opakuj:
   a) Vykresli_8_Symetrickych_Pixelu(x, y):
      [x, y], [y, x], [y, -x], [x, -y], [-x, -y], [-y, -x], [-y, x], [-x, y]
   
   b) Rozhodni o výšce Y a uprav prediktor P pro následující krok:
      POKUD je P < 0 PAK:
          // Středový bod je uvnitř kružnice -> jdeme ROVNĚ DOPRAVA (Y se nemění)
          P = P + 2 * x + 3
      JINAK (tedy P >= 0):
          // Středový bod je vně kružnice -> jdeme ŠIKMO DOLŮ (Y klesne o 1)
          P = P + 2 * (x - y) + 5
          y = y - 1
          
   c) Posuň se v ose X o 1 doprava:
      x = x + 1
```

In [4]:
display(vystup_grafu, ovladaci_panel)
vykresli_aktualni_stav()

Output(layout=Layout(min_height='680px'))